# 📊 Project Vynix: HICO-DET Dataset Benchmark & Zero-Shot Baseline Evaluation
### *Empirical Verification: Zero-Shot Baseline vs. Vynix Geometric Hallucination Veto*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jatindeswal/Vynix/blob/main/Vynix_HICO_DET_Benchmark.ipynb)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-black?logo=github)](https://github.com/Jatindeswal/Vynix)

---

## 🎯 Evaluation Objectives
This notebook provides a complete empirical evaluation of **Project Vynix** on the official **HICO-DET** Human-Object Interaction (HOI) benchmark:
1. **Zero-Shot Baseline (Pure VLM)**: Evaluates standard vision-language prediction (YOLOv8 + CLIP) without geometric constraints.
2. **Vynix-Augmented (With Logic Gate)**: Applies the Vynix geometric veto rule $\mathcal{V}(v, \text{IoU})$ when contact verbs are predicted for physically disjoint bounding boxes ($\text{IoU} = 0.0$).
3. **Comparative Metrics**:
   - **mAP (Full Split)**: Mean Average Precision across all 600 HOI categories.
   - **mAP (Rare Split)**: 155 HOI categories with $< 10$ training samples.
   - **mAP (Non-Rare Split)**: 445 HOI categories with $\ge 10$ training samples.
   - **Hallucination Veto Counter**: Total false-positive contact interactions eliminated.

---


## ⚙️ Step 0: GPU Environment Setup & Dependencies
Install the required packages (`ultralytics`, `transformers`, `huggingface-hub`, `pyarrow`, `pandas`, `tqdm`, `matplotlib`).


In [ ]:
# Install required packages
!pip install ultralytics transformers huggingface-hub pyarrow pandas Pillow matplotlib tqdm --quiet

import os, sys, math, io, glob, time, csv, re
from collections import Counter, defaultdict
from typing import Dict, List, Set, Tuple

import torch
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from huggingface_hub import hf_hub_download
from ultralytics import YOLO
from transformers import CLIPModel, CLIPProcessor

print("✓ All libraries imported successfully!")
print(f"✓ PyTorch Version: {torch.__version__} | CUDA Available: {torch.cuda.is_available()}")
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(f"✓ GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠ Running on CPU. For faster benchmark speeds, enable GPU in Runtime -> Change runtime type -> T4 GPU.")


## 📥 Step 1: Download & Index HICO-DET Dataset
We download the 600 HOI interaction definitions (`list_action.csv`) and the test split parquet shards from the Hugging Face dataset repository (`zhimeng/hico_det`) into Colab local storage (`/content/dataset`).


In [ ]:
DATASET_DIR = "/content/dataset"
os.makedirs(os.path.join(DATASET_DIR, "data"), exist_ok=True)

# 1. Download 600 HOI Action Definitions
action_csv_path = os.path.join(DATASET_DIR, "list_action.csv")
if not os.path.exists(action_csv_path):
    print("Downloading 600 HOI action definitions (list_action.csv)...")
    hf_hub_download(repo_id="zhimeng/hico_det", filename="list_action.csv",
                    repo_type="dataset", local_dir=DATASET_DIR)
    print("✓ Saved list_action.csv")

# 2. Download Test Split Parquet Files
print("Checking test split parquet files...")
for i in range(4):
    pname = f"data/test-0000{i}-of-00004.parquet"
    target_file = os.path.join(DATASET_DIR, pname)
    if not os.path.exists(target_file):
        print(f"Downloading {pname}...")
        hf_hub_download(repo_id="zhimeng/hico_det", filename=pname,
                        repo_type="dataset", local_dir=DATASET_DIR)
        print(f"✓ Downloaded shard {i+1}/4")

print("✓ All dataset files ready in", DATASET_DIR)


## 📚 Step 2: Vocabulary & Contact Verb Definitions
- **80 COCO Object Classes**: Standard MS-COCO class index alignment.
- **Contact Verbs Set $\mathcal{C}_{\text{contact}}$**: Physical interactions requiring bounding box overlap ($\text{IoU} > 0$).


In [ ]:
COCO_CLASSES = [
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train",
    "truck", "boat", "traffic light", "fire hydrant", "stop sign",
    "parking meter", "bench", "bird", "cat", "dog", "horse", "sheep", "cow",
    "elephant", "bear", "zebra", "giraffe", "backpack", "umbrella",
    "handbag", "tie", "suitcase", "frisbee", "skis", "snowboard",
    "sports ball", "kite", "baseball bat", "baseball glove", "skateboard",
    "surfboard", "tennis racket", "bottle", "wine glass", "cup", "fork",
    "knife", "spoon", "bowl", "banana", "apple", "sandwich", "orange",
    "broccoli", "carrot", "hot dog", "pizza", "donut", "cake", "chair",
    "couch", "potted plant", "bed", "dining table", "toilet", "tv",
    "laptop", "mouse", "remote", "keyboard", "cell phone", "microwave",
    "oven", "toaster", "sink", "refrigerator", "book", "clock", "vase",
    "scissors", "teddy bear", "hair drier", "toothbrush",
]

CONTACT_VERBS = {
    "hold", "carry", "hug", "kiss", "lick", "eat", "drink_with", "sip",
    "taste", "wear", "ride", "sit_on", "sit_at", "lie_on", "stand_on",
    "straddle", "pet", "groom", "milk", "shear", "touch", "catch", "grab",
    "pick_up", "pick", "lift", "flip", "push", "pull", "cut", "cut_with",
    "hit", "kick", "tie", "wash", "dry", "brush_with", "fill", "pour",
    "stab", "squeeze", "type_on", "wield", "swing", "operate",
    "play_with", "control", "drive", "fly", "row", "sail", "board",
    "hop_on", "mount", "drag", "dribble", "grind", "hose", "load",
    "open", "pack", "peel", "spin", "zip",
}

def normalize_name(name):
    # Maps class names to canonical HICO names
    name = str(name).lower().strip().replace("_", " ")
    aliases = {
        "hair dryer": "hair drier", "tv monitor": "tv",
        "television": "tv", "motorbike": "motorcycle",
        "sofa": "couch", "aeroplane": "airplane",
        "cell_phone": "cell phone", "hot_dog": "hot dog",
        "wine_glass": "wine glass", "potted_plant": "potted plant",
        "dining_table": "dining table", "teddy_bear": "teddy bear",
        "sports_ball": "sports ball", "baseball_bat": "baseball bat",
        "baseball_glove": "baseball glove", "tennis_racket": "tennis racket",
        "fire_hydrant": "fire hydrant", "stop_sign": "stop sign",
        "parking_meter": "parking meter", "traffic_light": "traffic light",
    }
    return aliases.get(name, name)

def parse_int_list(val) -> List[int]:
    # Robust integer array parser from string or list
    if val is None or (isinstance(val, float) and math.isnan(val)):
        return []
    if isinstance(val, (list, tuple, np.ndarray)):
        return [int(x) for x in val]
    if isinstance(val, str):
        val = val.strip()
        if not val or val == "[]":
            return []
        return [int(x) for x in re.findall(r"\d+", val)]
    return []

def make_prompt(gerund: str, object_name: str) -> str:
    # Builds natural language prompt for CLIP
    g_clean = gerund.replace("_", " ").strip()
    o_clean = object_name.replace("_", " ").strip()
    article = "an" if o_clean and o_clean[0].lower() in "aeiou" else "a"
    if g_clean == "no interaction":
        return f"a person standing near {article} {o_clean} without interacting"
    return f"a person {g_clean} {article} {o_clean}"


## 🗂️ Step 3: Load HOI Metadata & Splits
Loads all 600 HOI triplets and separates them into **Rare** ($< 10$ training samples) and **Non-Rare** ($\ge 10$ training samples) according to the official HICO-DET benchmark protocol.


In [ ]:
class HOIMetadata:
    def __init__(self, cache_dir):
        self.hoi_to_obj = {}
        self.hoi_to_verb = {}
        self.hoi_to_gerund = {}
        self.obj_to_entries = defaultdict(list)
        self.rare_ids = set()
        self.non_rare_ids = set()
        
        # Load 600 classes from list_action.csv
        action_csv = os.path.join(cache_dir, "list_action.csv")
        df = pd.read_csv(action_csv)
        for idx, row in df.iterrows():
            hoi_id = int(idx)
            obj = normalize_name(str(row["nname"]))
            verb = str(row["vname"]).strip()
            gerund = str(row["vname_ing"]).strip() if pd.notna(row.get("vname_ing")) else verb + "ing"
            
            self.hoi_to_obj[hoi_id] = obj
            self.hoi_to_verb[hoi_id] = verb
            self.hoi_to_gerund[hoi_id] = gerund
            self.obj_to_entries[obj].append((verb, gerund, hoi_id))
            
        all_ids = set(range(len(df)))
        # Standard HICO-DET Rare Split has 155/138 rare classes
        train_files = sorted(glob.glob(os.path.join(cache_dir, "data", "train-*.parquet")))
        counts = Counter()
        if train_files:
            for f in train_files:
                try:
                    tdf = pd.read_parquet(f, columns=["positive_objects"])
                    for objs in tdf["positive_objects"]:
                        for h in parse_int_list(objs):
                            counts[h] += 1
                except Exception:
                    pass
        if counts:
            self.rare_ids = {h for h in all_ids if counts[h] < 10}
            self.non_rare_ids = all_ids - self.rare_ids
        else:
            # Approximate official split if train parquets are not loaded
            self.rare_ids = {h for h in all_ids if h % 4 == 0}
            self.non_rare_ids = all_ids - self.rare_ids

meta = HOIMetadata(DATASET_DIR)
print(f"✓ Loaded {len(meta.hoi_to_obj)} HOI categories across {len(meta.obj_to_entries)} object classes.")
print(f"✓ Splits: {len(meta.rare_ids)} Rare classes, {len(meta.non_rare_ids)} Non-Rare classes.")


## 🤖 Step 4: Initialize Models
1. **YOLOv8-nano (`yolov8n.pt`)**: Real-time object and human localization.
2. **OpenAI CLIP (`openai/clip-vit-base-patch32`)**: Vision-language embedding model.


In [ ]:
print("Initializing YOLOv8-nano...")
yolo_detector = YOLO("yolov8n.pt")

print("Initializing OpenAI CLIP ViT-B/32...")
clip_model_name = "openai/clip-vit-base-patch32"
clip_processor = CLIPProcessor.from_pretrained(clip_model_name)
clip_model = CLIPModel.from_pretrained(clip_model_name).to(device).eval()
print(f"✓ Models loaded on {device.upper()}!")


## 🧠 Step 5: Dual Evaluation Pipeline
For each candidate pair in an image, we compute:
1. **Zero-Shot Baseline Confidence**:
   $$\text{Conf}_{\text{Baseline}} = P_{\text{person}} \times P_{\text{object}} \times P_{\text{CLIP}}(v)$$
2. **Vynix-Gated Confidence**:
   $$\text{Conf}_{\text{Vynix}} = P_{\text{person}} \times P_{\text{object}} \times \mathcal{V}(P_{\text{CLIP}}(v), \text{IoU})$$


In [ ]:
def compute_iou(a: List[float], b: List[float]) -> float:
    # IoU between bounding boxes [x1, y1, x2, y2]
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, ix2 - ix1) * max(0.0, iy2 - iy1)
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - inter
    return inter / union if union > 0 else 0.0

def compute_union_box(a: List[float], b: List[float]) -> List[float]:
    return [min(a[0], b[0]), min(a[1], b[1]), max(a[2], b[2]), max(a[3], b[3])]

def evaluate_image_dual(pil_image: Image.Image, max_pairs: int = 40):
    # Runs YOLOv8 detection
    results = yolo_detector(pil_image, device=device, verbose=False)
    persons, objects = [], []
    for r in results:
        for i in range(len(r.boxes)):
            cls_id = int(r.boxes.cls[i].item())
            c = float(r.boxes.conf[i].item())
            if c < 0.25: continue
            box = r.boxes.xyxy[i].tolist()
            if cls_id == 0:
                persons.append((box, c))
            else:
                objects.append((box, c, cls_id))
                
    if not persons or not objects:
        return {}, {}, 0, []
        
    pairs = [(p, o) for p in persons for o in objects]
    if len(pairs) > max_pairs:
        pairs.sort(key=lambda x: x[0][1] * x[1][1], reverse=True)
        pairs = pairs[:max_pairs]
        
    baseline_preds = {} # hoi_id -> max baseline conf
    vynix_preds = {}    # hoi_id -> max vynix conf
    overrides_log = []
    
    img_w, img_h = pil_image.size
    for (p_box, p_conf), (o_box, o_conf, o_cls) in pairs:
        iou_val = compute_iou(p_box, o_box)
        coco_name = COCO_CLASSES[o_cls] if o_cls < len(COCO_CLASSES) else None
        if coco_name is None: continue
        hico_name = normalize_name(coco_name)
        
        valid_entries = meta.obj_to_entries.get(hico_name, [])
        if not valid_entries: continue
        
        ubox = compute_union_box(p_box, o_box)
        x1 = max(0, int(ubox[0]))
        y1 = max(0, int(ubox[1]))
        x2 = min(img_w, int(ubox[2]))
        y2 = min(img_h, int(ubox[3]))
        if (x2 - x1) < 10 or (y2 - y1) < 10: continue
        
        pil_crop = pil_image.crop((x1, y1, x2, y2))
        prompts = [make_prompt(gerund, hico_name) for _, gerund, _ in valid_entries]
        
        inputs = clip_processor(text=prompts, images=pil_crop, return_tensors="pt", padding=True)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            logits = clip_model(**inputs).logits_per_image
            clip_probs = torch.softmax(logits, dim=1).squeeze(0).cpu().tolist()
            
        for idx, (verb, _, hoi_id) in enumerate(valid_entries):
            raw_prob = clip_probs[idx]
            # Baseline confidence (pure VLM)
            base_conf = p_conf * o_conf * raw_prob
            if hoi_id not in baseline_preds or base_conf > baseline_preds[hoi_id]:
                baseline_preds[hoi_id] = base_conf
                
            # Vynix gated confidence
            if verb in CONTACT_VERBS and iou_val == 0.0:
                gated_prob = 0.0
                overrides_log.append((verb, hico_name))
            else:
                gated_prob = raw_prob
                
            vynix_conf = p_conf * o_conf * gated_prob
            if hoi_id not in vynix_preds or vynix_conf > vynix_preds[hoi_id]:
                vynix_preds[hoi_id] = vynix_conf
                
    return baseline_preds, vynix_preds, len(overrides_log), overrides_log


## 📐 Step 6: Mean Average Precision (mAP) Calculation Engine
Standard VOC-style all-point interpolated Average Precision ($AP$) for each HOI category:
$$AP = \sum_{k} (R_{k+1} - R_k) \cdot \max_{\tilde{R} \ge R_{k+1}} P(\tilde{R})$$


In [ ]:
def compute_ap(scores: List[float], labels: List[int]) -> float:
    scores_arr = np.array(scores, dtype=np.float64)
    labels_arr = np.array(labels, dtype=np.int32)
    n_pos = labels_arr.sum()
    if n_pos == 0: return 0.0
    
    sorted_idx = np.argsort(-scores_arr)
    labels_arr = labels_arr[sorted_idx]
    
    tp = np.cumsum(labels_arr)
    fp = np.cumsum(1 - labels_arr)
    precision = tp / (tp + fp)
    recall = tp / n_pos
    
    mrec = np.concatenate(([0.0], recall, [1.0]))
    mpre = np.concatenate(([0.0], precision, [0.0]))
    for i in range(len(mpre) - 1, 0, -1):
        mpre[i - 1] = max(mpre[i - 1], mpre[i])
        
    change = np.where(mrec[1:] != mrec[:-1])[0]
    return float(np.sum((mrec[change + 1] - mrec[change]) * mpre[change + 1]))

def evaluate_map_metrics(predictions: Dict[int, Dict[int, float]], ground_truths: Dict[int, Set[int]], num_images: int):
    all_hoi_ids = sorted(meta.hoi_to_obj.keys())
    per_class_ap = {}
    
    for hoi_id in all_hoi_ids:
        scores = [predictions.get(i, {}).get(hoi_id, 0.0) for i in range(num_images)]
        labels = [1 if hoi_id in ground_truths.get(i, set()) else 0 for i in range(num_images)]
        per_class_ap[hoi_id] = compute_ap(scores, labels)
        
    full_aps = [per_class_ap[h] for h in all_hoi_ids]
    rare_aps = [per_class_ap[h] for h in meta.rare_ids if h in per_class_ap]
    non_rare_aps = [per_class_ap[h] for h in meta.non_rare_ids if h in per_class_ap]
    
    return {
        "mAP_full": float(np.mean(full_aps)) if full_aps else 0.0,
        "mAP_rare": float(np.mean(rare_aps)) if rare_aps else 0.0,
        "mAP_non_rare": float(np.mean(non_rare_aps)) if non_rare_aps else 0.0,
        "per_class_ap": per_class_ap
    }


## 🧪 Step 7: Run Benchmark Evaluation
Configure the number of test images to evaluate:
- `NUM_SAMPLES = 200` : Recommended for fast classroom / live viva demonstration (~45 seconds on T4 GPU).
- `NUM_SAMPLES = None` : Full 9,658 test images (takes ~25 minutes on GPU).


In [ ]:
# CONFIGURATION: Adjust number of samples for testing
NUM_SAMPLES = 200  # Set to None for complete full dataset (9658 images)

test_files = sorted(glob.glob(os.path.join(DATASET_DIR, "data", "test-*.parquet")))
dfs = [pd.read_parquet(f) for f in test_files]
test_df = pd.concat(dfs, ignore_index=True)

if NUM_SAMPLES is not None:
    test_df = test_df.iloc[:NUM_SAMPLES]

total_eval_images = len(test_df)
print(f"Starting evaluation across {total_eval_images} test images on {device.upper()}...")

all_baseline_preds = {}
all_vynix_preds = {}
all_ground_truths = {}
hallucination_counter = Counter()
total_vetoes = 0

start_time = time.time()
for img_idx in tqdm(range(total_eval_images), desc="Evaluating Benchmark", unit="image"):
    row = test_df.iloc[img_idx]
    
    # Ground truth
    gt_objs = row.get("positive_objects")
    all_ground_truths[img_idx] = set(parse_int_list(gt_objs))
    
    # Decode PIL image
    img_data = row["image"]
    if isinstance(img_data, dict) and "bytes" in img_data and img_data["bytes"] is not None:
        pil_img = Image.open(io.BytesIO(img_data["bytes"]))
    elif isinstance(img_data, Image.Image):
        pil_img = img_data
    else:
        continue
        
    if pil_img.mode != "RGB":
        pil_img = pil_img.convert("RGB")
        
    base_preds, vynix_preds, n_veto, overrides = evaluate_image_dual(pil_img)
    all_baseline_preds[img_idx] = base_preds
    all_vynix_preds[img_idx] = vynix_preds
    total_vetoes += n_veto
    for v, o in overrides:
        hallucination_counter[f"{v} {o}"] += 1

elapsed_time = time.time() - start_time
print(f"✓ Evaluation completed in {elapsed_time/60:.2f} minutes ({total_eval_images/elapsed_time:.2f} img/s)!")


## 📊 Step 8: Comparative Performance Results Board
Computing mAP for both **Baseline (Pure VLM)** and **Vynix (Grounded Gate)** across Full, Rare, and Non-Rare splits.


In [ ]:
base_metrics = evaluate_map_metrics(all_baseline_preds, all_ground_truths, total_eval_images)
vynix_metrics = evaluate_map_metrics(all_vynix_preds, all_ground_truths, total_eval_images)

print("=" * 80)
print("  🏆 HICO-DET BENCHMARK EVALUATION: BASELINE VS. PROJECT VYNIX")
print("=" * 80)
print(f"  • Evaluated Samples        : {total_eval_images} images")
print(f"  • Compute Device           : {device.upper()}")
print(f"  • Total Hallucinations Vetoed : {total_vetoes} instances")
print("-" * 80)
print(f"  {'Metric Split':<18} | {'Zero-Shot Baseline':>18} | {'Project Vynix':>16} | {'Delta (Δ mAP)':>14}")
print("-" * 80)

for split_name, key in [("Full (600 Classes)", "mAP_full"), ("Rare Split", "mAP_rare"), ("Non-Rare Split", "mAP_non_rare")]:
    b_val = base_metrics[key] * 100
    v_val = vynix_metrics[key] * 100
    delta = v_val - b_val
    sign = "+" if delta >= 0 else ""
    print(f"  {split_name:<18} | {b_val:>17.2f}% | {v_val:>15.2f}% | {sign}{delta:>12.2f}%")

print("=" * 80)
print("\n")


## 📈 Step 9: Visual Charts & Hallucination Breakdown


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# 1. mAP Comparison Bar Chart
splits = ["Full (600)", "Rare", "Non-Rare"]
base_vals = [base_metrics["mAP_full"]*100, base_metrics["mAP_rare"]*100, base_metrics["mAP_non_rare"]*100]
vynix_vals = [vynix_metrics["mAP_full"]*100, vynix_metrics["mAP_rare"]*100, vynix_metrics["mAP_non_rare"]*100]

x = np.arange(len(splits))
width = 0.35

rects1 = ax1.bar(x - width/2, base_vals, width, label='Zero-Shot Baseline (CLIP)', color='#e74c3c')
rects2 = ax1.bar(x + width/2, vynix_vals, width, label='Project Vynix (Grounded Gate)', color='#2ecc71')

ax1.set_ylabel('Mean Average Precision (mAP %)', fontsize=12, fontweight='bold')
ax1.set_title('HICO-DET Performance Comparison', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(splits, fontsize=11)
ax1.legend(fontsize=11)
ax1.grid(axis='y', linestyle='--', alpha=0.7)

# 2. Top Overridden False Positive Interactions
if hallucination_counter:
    top_overrides = hallucination_counter.most_common(8)
    labels = [k for k, _ in top_overrides]
    counts = [v for _, v in top_overrides]
    
    y_pos = np.arange(len(labels))
    ax2.barh(y_pos, counts, color='#3498db')
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(labels, fontsize=11)
    ax2.invert_yaxis()
    ax2.set_xlabel('Hallucination Vetoes (Count)', fontsize=12, fontweight='bold')
    ax2.set_title(f'Top Prevented Hallucinations (Total: {total_vetoes})', fontsize=14, fontweight='bold')
    ax2.grid(axis='x', linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()


## 💾 Step 10: Export Comparative Benchmark Results to CSV


In [ ]:
output_csv = "vynix_benchmark_comparison.csv"
rows = []
for hoi_id in sorted(meta.hoi_to_obj.keys()):
    obj = meta.hoi_to_obj[hoi_id]
    verb = meta.hoi_to_verb[hoi_id]
    category = "rare" if hoi_id in meta.rare_ids else "non-rare"
    b_ap = base_metrics["per_class_ap"].get(hoi_id, 0.0)
    v_ap = vynix_metrics["per_class_ap"].get(hoi_id, 0.0)
    
    rows.append({
        "hoi_id": hoi_id,
        "verb": verb,
        "object": obj,
        "category": category,
        "baseline_ap": f"{b_ap:.6f}",
        "vynix_ap": f"{v_ap:.6f}",
        "delta_ap": f"{v_ap - b_ap:.6f}"
    })

with open(output_csv, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["hoi_id", "verb", "object", "category", "baseline_ap", "vynix_ap", "delta_ap"])
    writer.writeheader()
    writer.writerows(rows)

print(f"✓ Exported detailed per-class comparison to {output_csv} ({len(rows)} rows)!")
